NDCG (Normalized Discounted Cumulative Gain，归一化折损累计收益)

第一步：CG (Cumulative Gain，累计收益)

既然不能只看第一个，那我们就把排行榜前 $K$ 个物品的 **相关性得分（Relevance Score, $rel$）** 全部加起来。
- 注：这里的相关性不再是简单的 0 和 1。比如极其相关是 3，一般相关是 1，不相关是 0。
- 缺陷：它没考虑“位置”的影响。把相关性为 3 的物品排在第 1 位和排在第 10 位，CG 算出来的总分是一样的。但用户显然更希望好东西排在前面。


第二步：DCG (Discounted CG，折损累计收益)

为了让模型懂得“好东西必须靠前排”，我们引入了位置惩罚（Discount）。物品排得越靠后，用户滑到那里的耐心就越少，所以它的价值要打折扣。数学上通常用对数衰减来模拟用户的耐心消耗：$$DCG_K = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i+1)}$$
- 分母 $\log_2(i+1)$ 随着排名位置 $i$ 的增加而变大，导致排在后面的物品得出的真实收益急剧下降。这就逼着模型把高分物品顶上去。


第三步：IDCG (Ideal DCG，理想情况下的 DCG)

为什么要这一步？因为不同用户的“绝对分数”不可比。
- 用户 A 搜了一个词，库里有 10 个极其相关的结果，模型排得很好，算出来的 DCG 是 15。
- 用户 B 搜了一个极其生僻的词，库里只有 1 个勉强相关的结果，模型就算排在第 1 位，算出来的 DCG 也只有 2。
- 我们能说模型对用户 A 的服务比对用户 B 好吗？不能！因为那是数据丰富度决定的。为了跨用户/跨查询进行公平比较，我们需要一个“理论最高分”。
- IDCG：假设我们有一个开了“上帝视角”的完美模型，它把所有候选物品严格按照相关性从大到小排序。在这个完美排行榜下算出的 DCG，就是 IDCG。


第四步：NDCG (Normalized DCG，归一化)

最后，我们把模型真实给出的 DCG，除以那个理想的完美分数 IDCG，就得到了归一化的 NDCG：$$NDCG_K = \frac{DCG_K}{IDCG_K}$$
- 这样一来，无论对于用户 A 还是用户 B，算出来的 NDCG 都在 0 到 1 之间。1.0 就代表模型排出了完美的序列。我们可以放心地把所有用户的 NDCG 加起来求平均了。

In [6]:
import math

def dcg_k(sorted_y_true, k):
    y_trues = sorted_y_true[:k]
    
    dcg_sum = 0.0
    for idx, y_true in enumerate(y_trues):
        sum += y_true / math.log2(2 + idx)

    return dcg_sum

def ndcg_k(y_true, y_pred, k):
    sorted_y_pair = list(sorted(zip(y_true, y_pred), key=lambda x:-x[1]))
    sorted_y_true, _ = zip(*sorted_y_pair)
    dcg = dcg_k(sorted_y_true, k)

    ideal_y_true = sorted(y_true, reverse=True)
    idcg = dcg_k(ideal_y_true, k)

    if idcg == 0.0:
        return 0.0

    return dcg / idcg

y_true = [3, 1, 2, 0] # 真实相关性 (3最高)
y_pred = [0.9, 0.1, 0.8, 0.3] # 模型打分
print("NDCG@2:", ndcg_k(y_true, y_pred, k=2))

NDCG@2: 1.0


In [4]:
import math

def dcg_k(sorted_labels, k):
    sorted_labels = list(sorted_labels)[:k]
    dcg = 0.0
    for idx, label in enumerate(sorted_labels):
        dcg += label / math.log2(2 + idx)
    return dcg

def ndcg_k(y_true, y_pred, k):
    true_order, _ = zip(*sorted(zip(y_true, y_pred), key=lambda x:-x[1]))
    dcg = dcg_k(true_order, k)

    ideal_order = sorted(y_true, reverse=True)
    idcg = dcg_k(ideal_order, k)

    if idcg == 0.0:
        return 0.0
    
    return dcg / idcg

y_true = [3, 1, 2, 0] # 真实相关性 (3最高)
y_pred = [0.9, 0.1, 0.8, 0.3] # 模型打分
print("NDCG@2:", ndcg_k(y_true, y_pred, k=2))

NDCG@2: 1.0
